# NB-01 | Repo Ingestion & Surface Extraction
**Pipeline Step 1 of 5**

Fetches the repository file tree from the GitHub API, selects security-relevant files, optionally parses an OpenAPI spec, and writes `repo_surface.json` for downstream steps.

**Key improvements over v1:**
- `valid_paths` stores full relative paths (not just basenames) so the hallucination validator in NB03/04/05 can do exact-path matching instead of substring matching
- Rate-limit detection and exponential back-off live in `pipeline_utils.github_get` — not duplicated here
- File-selection caps are enforced correctly and are independently configurable
- Output includes a `run_id` (repo + timestamp) used by all downstream steps to namespace archives

**Output:** `repo_surface.json`

In [ ]:
!pip install -q requests PyYAML

import os, sys
sys.path.insert(0, ".")

from pipeline_utils import github_get, save_json, get_logger

import json, re, time, base64, datetime
import requests
import yaml
from pathlib import Path
from collections import defaultdict

log = get_logger("NB01")

## 1.1 — Configuration

Edit `TARGET_REPO` and optionally set `GITHUB_TOKEN` in your environment.

In [ ]:
# ── Required ──────────────────────────────────────────────────────────────────
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
TARGET_REPO  = os.environ.get("TARGET_REPO", "tiangolo/fastapi")
OUTPUT_PATH  = "repo_surface.json"

# ── File selection ────────────────────────────────────────────────────────────
PRIORITY_PATTERNS = [
    r"route", r"controller", r"handler", r"auth", r"model",
    r"schema", r"middleware", r"api", r"view", r"endpoint",
    r"service", r"repository", r"db", r"database", r"config",
]
PRIORITY_EXTS     = {".py", ".ts", ".js", ".go", ".java", ".rb", ".php"}
MAX_FILES_PER_EXT = 15      # cap per extension to prevent one language dominating
HARD_FILE_CAP     = 60      # absolute ceiling across all extensions
MAX_FILE_CHARS    = 3_000   # characters to keep per file

# ── OpenAPI candidate paths (checked in order) ────────────────────────────────
OPENAPI_CANDIDATES = [
    "openapi.yaml", "openapi.json", "swagger.yaml", "swagger.json",
    "api/openapi.yaml", "docs/openapi.yaml", "spec/openapi.yaml",
    "src/openapi.yaml", "schema/openapi.yaml",
]

GH_HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

# Unique identifier for this pipeline run — used to namespace archived outputs
RUN_ID = f"{TARGET_REPO.replace('/', '__')}__{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"

log.info("Targeting repo : %s", TARGET_REPO)
log.info("Authenticated  : %s", bool(GITHUB_TOKEN))
log.info("Run ID         : %s", RUN_ID)

## 1.2 — Fetch Repository Tree

In [ ]:
def fetch_repo_tree(repo: str) -> list[dict]:
    """
    Recursively fetch all file blobs via the GitHub Trees API.

    Warns if the tree is truncated (repo > ~100k files).
    Returns a list of blob dicts, each with at minimum 'path' and 'type'.
    """
    url  = f"https://api.github.com/repos/{repo}/git/trees/HEAD?recursive=1"
    resp = github_get(url, GH_HEADERS, logger=log)
    if resp is None:
        raise ValueError(f"Repo not found or inaccessible: {repo}")
    data = resp.json()
    if data.get("truncated"):
        log.warning("Tree response is truncated — very large repo; some files will be missed")
    blobs = [item for item in data.get("tree", []) if item["type"] == "blob"]
    log.info("Total blobs in tree: %d", len(blobs))
    return blobs

tree      = fetch_repo_tree(TARGET_REPO)
all_paths = {item["path"] for item in tree}   # set for O(1) lookup

## 1.3 — Priority File Selection

Files are scored by how many security-relevant name patterns they match plus a bonus for being in a security-relevant extension set. We then cap per extension to prevent one language monopolising the context budget.

In [ ]:
def priority_score(path: str) -> int:
    """Score a file path by relevance to security-sensitive code."""
    path_lower = path.lower()
    score = sum(1 for pat in PRIORITY_PATTERNS if re.search(pat, path_lower))
    if Path(path).suffix in PRIORITY_EXTS:
        score += 1
    return score


# Score and sort all blobs
scored_blobs = [
    (item["path"], priority_score(item["path"]), item)
    for item in tree
    if Path(item["path"]).suffix in PRIORITY_EXTS
]
scored_blobs.sort(key=lambda x: x[1], reverse=True)

# Cap per extension
ext_buckets: dict[str, list] = defaultdict(list)
for path, score, item in scored_blobs:
    ext = Path(path).suffix
    if len(ext_buckets[ext]) < MAX_FILES_PER_EXT:
        ext_buckets[ext].append((path, score, item))

# Flatten and apply hard cap, maintaining priority order
all_selected: list[tuple] = []
for items in ext_buckets.values():
    all_selected.extend(items)
all_selected.sort(key=lambda x: x[1], reverse=True)
selected_files = all_selected[:HARD_FILE_CAP]

log.info("Selected %d priority files (cap=%d)", len(selected_files), HARD_FILE_CAP)
print("\nTop 10 by priority score:")
for path, score, _ in selected_files[:10]:
    print(f"  [{score:2d}] {path}")

## 1.4 — Fetch File Contents

In [ ]:
def fetch_file_content(repo: str, path: str) -> str:
    """
    Download and base64-decode a single file from the GitHub Contents API.

    Returns the decoded text (truncated to MAX_FILE_CHARS) or an empty string
    if the file is inaccessible or binary. Logs the specific failure reason.
    """
    url  = f"https://api.github.com/repos/{repo}/contents/{path}"
    resp = github_get(url, GH_HEADERS, logger=log)
    if resp is None:
        log.warning("Skipping %s — file not accessible (404)", path)
        return ""

    data = resp.json()
    if data.get("encoding") != "base64":
        log.warning("Skipping %s — unexpected encoding: %s", path, data.get("encoding"))
        return ""

    try:
        return base64.b64decode(data["content"]).decode("utf-8", errors="replace")[:MAX_FILE_CHARS]
    except Exception as exc:
        log.warning("Decode error for %s: %s", path, exc)
        return ""


file_contents: dict[str, str] = {}
fetch_errors: list[str] = []

for path, score, item in selected_files:
    content = fetch_file_content(TARGET_REPO, path)
    if content.strip():
        file_contents[path] = content
    else:
        fetch_errors.append(path)
    time.sleep(0.1)   # polite rate-limit buffer

log.info("Fetched: %d files | Skipped (empty/error): %d", len(file_contents), len(fetch_errors))
log.info("Total chars: %s", f"{sum(len(v) for v in file_contents.values()):,}")

## 1.5 — OpenAPI Detection & Parsing

In [ ]:
def fetch_openapi(repo: str, tree_paths: set[str]) -> dict | None:
    """
    Locate and parse the first valid OpenAPI/Swagger spec found in the repo tree.

    Tries a ranked list of candidate paths. Returns the parsed spec dict, or
    None if no valid spec is found.
    """
    for candidate in OPENAPI_CANDIDATES:
        if candidate not in tree_paths:
            continue
        raw = fetch_file_content(repo, candidate)
        if not raw:
            continue
        try:
            if candidate.endswith(".yaml"):
                spec = yaml.safe_load(raw)
            else:
                spec = json.loads(raw)
            # Minimal validity check — must look like an OpenAPI document
            if isinstance(spec, dict) and any(k in spec for k in ("paths", "openapi", "swagger")):
                log.info("Found OpenAPI spec at: %s", candidate)
                return spec
            else:
                log.warning("%s parsed but does not look like an OpenAPI doc — skipping", candidate)
        except Exception as exc:
            log.warning("Could not parse %s: %s", candidate, exc)
    return None


openapi_spec = fetch_openapi(TARGET_REPO, all_paths)

if openapi_spec:
    version   = openapi_spec.get("openapi", openapi_spec.get("swagger", "unknown"))
    endpoints = list(openapi_spec.get("paths", {}).keys())
    schemas   = list(openapi_spec.get("components", {}).get("schemas", {}).keys())
    log.info("OpenAPI version: %s | Endpoints: %d | Schemas: %d", version, len(endpoints), len(schemas))
    print("\nEndpoint sample:")
    for ep in endpoints[:8]:
        print(f"  {ep}")
else:
    log.info("No OpenAPI spec found — surface will be inferred from code only")

## 1.6 — Build & Save `repo_surface.json`

`valid_paths` stores **full relative paths** (e.g. `fastapi/routing.py`), not just basenames. Downstream hallucination validators use these for exact-path matching, falling back to basename-only matching when evidence omits directory prefixes.

In [ ]:
surface = {
    "repo":        TARGET_REPO,
    "run_id":      RUN_ID,
    "file_count":  len(file_contents),
    "files":       file_contents,          # {full_path: content}
    "valid_paths": sorted(file_contents.keys()),   # full relative paths for exact matching
    "openapi":     openapi_spec,
    "tree_sample": sorted(all_paths)[:200],
    "metadata": {
        "total_tree_blobs":    len(tree),
        "selected_file_count": len(selected_files),
        "fetch_errors":        fetch_errors,
        "authenticated":       bool(GITHUB_TOKEN),
        "max_file_chars":      MAX_FILE_CHARS,
        "hard_file_cap":       HARD_FILE_CAP,
    },
}

save_json(surface, OUTPUT_PATH, run_id=RUN_ID)

print("\n=== SURFACE SUMMARY ===")
print(f"  Repo         : {TARGET_REPO}")
print(f"  Run ID       : {RUN_ID}")
print(f"  Files fetched: {len(file_contents)}")
print(f"  Valid paths  : {len(surface['valid_paths'])}")
print(f"  Has OpenAPI  : {openapi_spec is not None}")
print(f"  Total chars  : {sum(len(v) for v in file_contents.values()):,}")
print(f"  Fetch errors : {len(fetch_errors)}")